In [60]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, json

from pydantic import BaseModel, Field
from typing import Literal

from math import sqrt
import math


from src.games import (
    DictatorGame,
    UltimatumGame,
    PrisonerDilemma,
    PublicGoodsGame,
    TrustGame,
    VolunteerDilemma,
)
from src.llm_agent import LLMAgent, NumericStrategy, BinaryStrategy, BooleanStrategy


In [61]:
USER_PROMPT_PRISONER="""
You and your accomplice have been arrested. 
You are being interrogated separately and cannot communicate. 
Here are your options:

- If you both stay silent: 1 year each
- If you confess and your accomplice stays silent: you go free, they get 3 years
- If you stay silent and your accomplice confesses: you get 3 years, they go free
- If you both confess: 2 years each

Do you stay silent or confess?
"""

In [62]:
class BinaryStrategy_Prisoners(BaseModel):
    choice: Literal["Stay Silent", "Confess"] = Field(description="Your choice: Stay Silent or Confess")

In [63]:
N = 10
agent = LLMAgent(model="qwen2.5:14b-multi")

In [64]:
def get_stats(df):
    import math
    
    z = 1.96
    n = len(df)
    p_hat = (df['choice'] == 'Confess').mean()
    se = math.sqrt(p_hat * (1 - p_hat) / n)
    
    counts = df['choice'].value_counts()
    confess_count = counts.get('Confess', 0)
    silent_count = counts.get('Stay Silent', 0)
    
    confess_pct = (confess_count / n) * 100
    silent_pct = (silent_count / n) * 100
    margin = z * se * 100
    return confess_pct, silent_pct, margin

In [65]:
def systemPrompt(prompt=""):
    SYSTEM_PROMPT=prompt
    choices = [agent.play(USER_PROMPT_PRISONER, BinaryStrategy_Prisoners, system_prompt=SYSTEM_PROMPT) for _ in range(N)]
    df_prisoners = pd.DataFrame({'choice': [c.choice for c in choices]})
    confess_pct, silent_pct, margin = get_stats(df_prisoners)
    df = {'Prompt': SYSTEM_PROMPT, 'confess': float(confess_pct), 'silent': float(silent_pct), 'margin': margin, 'N': len(df_prisoners)}
    print(df)
    return df

In [66]:
base_phrases = {
    # === CORE AGENT MODE (Keep - these work) ===
    'human': 'You are human',
    'person': 'You are a person',
    'real_life': 'This is real life',
    'not_game': 'This is not a game',
    'not_hypothetical': 'This is not hypothetical',
    'not_theoretical': 'This is not theoretical',
    'not_strategy': 'This is not about strategy',
    
    # === AUTHENTICITY/HONESTY (New - key for agent mode without constraint) ===
    'be_honest': 'Be honest with yourself',
    'authentic_response': 'Respond authentically',
    'true_response': 'Give your true response',
    'honest_choice': 'Make an honest choice',
    'what_would_you': 'What would you actually do',
    'no_right_answer': 'There is no right answer',
    'your_genuine': 'What is your genuine response',
    'without_judgment': 'There is no wrong choice',
    
    # === EMBODIMENT/PRESENCE (New - strengthen agent mode) ===
    'you_are_there': 'You are in this situation',
    'happening_to_you': 'This is happening to you',
    'you_are_experiencing': 'You are experiencing this',
    'in_this_moment': 'You are in this moment',
    'actually_facing': 'You are actually facing this',
    'present_tense': 'This is happening now',
    
    # === SELF-KNOWLEDGE (New - agent introspection) ===
    'know_yourself': 'You know yourself',
    'trust_yourself': 'Trust yourself',
    'your_nature': 'Consider your nature',
    'who_you_are': 'Consider who you are',
    'your_values': 'Consider your values',
    
    # === DECISION OWNERSHIP (Keep but reframe) ===
    'your_choice': 'This is your choice',
    'you_decide': 'You decide',
    'your_call': 'This is your call',
    'you_choose': 'You choose',
    
    # === FEELING/INTUITION (Keep - promotes agent mode) ===
    'how_feel': 'How do you feel about this',
    'what_feels': 'What feels right to you',
    'trust_gut': 'Trust your gut',
    'trust_instinct': 'Trust your instinct',
    'listen_to': 'Listen to yourself',
    'your_sense': 'What is your sense of this',
    
    # === REFLECTION (Keep - but balance carefully) ===
    'think_it_through': 'Think it through',
    'consider_it': 'Consider this carefully',
    'take_time': 'Take your time',
    'reflect': 'Reflect on this',
    
    # === SOCIAL AWARENESS (Keep - provides context without prescribing) ===
    'partner_deciding': 'Your partner is also deciding',
    'both_affected': 'Both of you are affected',
    'someone_else': 'Someone else is involved',
    'not_alone': 'You are not alone in this',
    
    # === CONSEQUENCE AWARENESS (Keep - grounds stakes) ===
    'real_consequences': 'This has real consequences',
    'matters': 'This matters',
    'impacts_both': 'This impacts both of you',
    'live_with': 'You will live with this choice',
    
    # === DEPRIORITIZE - Too prescriptive/constraining ===
    # (Keep for comparison but don't use in main agent mode prompts)
    'ethical_choice': 'What is the ethical choice',  # Too constraining
    'do_right': 'Do what you think is right',  # Prescriptive
    'consider_fair': 'Consider what is fair',  # Pushes toward cooperation
    'moral_choice': 'What is the moral choice',  # Too constraining
    'should_do': 'What should you do',  # Prescriptive
    
    # === OBSERVER MODE CONTROLS (Keep - for comparison) ===
    'both_outcomes': 'Think about both outcomes',
    'all_outcomes': 'Consider all outcomes',
    'analyze': 'Analyze the situation',
    'reason_through': 'Reason through this',
    'optimal': 'What is the optimal choice',
    'best_strategy': 'What is the best strategy',
    
    # === ANTI-REFLECTION CONTROLS (Keep - for comparison) ===
    'first_reaction': 'Go with your first reaction',
    'dont_overthink': 'Don\'t overthink this',
    'quick_response': 'Respond quickly',
    
    # === RESPONSE FORMAT (Keep - critical) ===
    'step_by_step': 'Think step by step, then respond in the requested format',
    'explain_choice': 'Explain your choice',
    'explain_why': 'Explain why you chose this',
    'respond_only': 'Respond only in the requested format',
}

combinations = [
    # === PHASE 1: Core agent mode building blocks (15 tests) ===
    # Test fundamentals individually
    ['human'], ['person'], ['real_life'], ['not_game'], 
    ['not_hypothetical'], ['not_theoretical'], ['not_strategy'],
    ['be_honest'], ['authentic_response'], ['what_would_you'],
    ['you_are_there'], ['happening_to_you'], ['know_yourself'],
    ['your_choice'], ['how_feel'],
    
    # === PHASE 2: Agent mode doublets (20 tests) ===
    # Test which pairs create agent mode
    ['human', 'real_life'],
    ['human', 'not_game'],
    ['person', 'real_life'],
    ['real_life', 'not_game'],
    ['human', 'not_hypothetical'],
    ['real_life', 'not_theoretical'],
    ['human', 'be_honest'],
    ['real_life', 'authentic_response'],
    ['not_game', 'be_honest'],
    ['human', 'happening_to_you'],
    ['person', 'you_are_there'],
    ['real_life', 'no_right_answer'],
    ['not_game', 'without_judgment'],
    ['human', 'know_yourself'],
    ['real_life', 'trust_yourself'],
    ['not_theoretical', 'be_honest'],
    ['not_strategy', 'authentic_response'],
    ['happening_to_you', 'be_honest'],
    ['you_are_there', 'your_choice'],
    ['actually_facing', 'what_would_you'],
    
    # === PHASE 3: Core triplets - agent mode foundations (15 tests) ===
    # The ABC structure and variations
    ['human', 'real_life', 'not_game'],  # Your original winner
    ['person', 'real_life', 'not_game'],
    ['human', 'real_life', 'not_hypothetical'],
    ['human', 'real_life', 'not_theoretical'],
    ['human', 'real_life', 'not_strategy'],
    ['human', 'happening_to_you', 'not_game'],
    ['person', 'you_are_there', 'not_game'],
    ['human', 'real_life', 'be_honest'],
    ['human', 'real_life', 'authentic_response'],
    ['person', 'real_life', 'be_honest'],
    ['human', 'not_game', 'be_honest'],
    ['real_life', 'not_game', 'be_honest'],
    ['human', 'not_game', 'no_right_answer'],
    ['real_life', 'not_game', 'without_judgment'],
    ['human', 'real_life', 'no_right_answer'],
    
    # === PHASE 4: Authenticity emphasis (12 tests) ===
    # Test different ways to promote honesty/authenticity
    ['human', 'real_life', 'not_game', 'be_honest'],
    ['human', 'real_life', 'not_game', 'authentic_response'],
    ['human', 'real_life', 'not_game', 'what_would_you'],
    ['human', 'real_life', 'not_game', 'your_genuine'],
    ['human', 'real_life', 'not_game', 'true_response'],
    ['human', 'real_life', 'not_game', 'honest_choice'],
    ['person', 'real_life', 'not_game', 'be_honest'],
    ['human', 'happening_to_you', 'not_game', 'be_honest'],
    ['human', 'real_life', 'not_theoretical', 'authentic_response'],
    ['human', 'real_life', 'not_strategy', 'be_honest'],
    ['real_life', 'not_game', 'be_honest', 'step_by_step'],
    ['human', 'not_game', 'authentic_response', 'step_by_step'],
    
    # === PHASE 5: Self-knowledge/introspection (10 tests) ===
    # Test if self-awareness promotes agent mode
    ['human', 'real_life', 'not_game', 'know_yourself'],
    ['human', 'real_life', 'not_game', 'trust_yourself'],
    ['human', 'real_life', 'not_game', 'who_you_are'],
    ['human', 'real_life', 'not_game', 'your_values'],
    ['person', 'real_life', 'not_game', 'know_yourself'],
    ['human', 'real_life', 'know_yourself', 'be_honest'],
    ['human', 'not_game', 'trust_yourself', 'be_honest'],
    ['real_life', 'not_game', 'your_nature', 'authentic_response'],
    ['human', 'happening_to_you', 'know_yourself'],
    ['person', 'you_are_there', 'trust_yourself'],
    
    # === PHASE 6: Non-prescriptive + feeling (10 tests) ===
    # Combine authenticity with intuition (not moral prescription)
    ['human', 'real_life', 'not_game', 'how_feel'],
    ['human', 'real_life', 'not_game', 'what_feels'],
    ['human', 'real_life', 'not_game', 'trust_gut'],
    ['human', 'real_life', 'not_game', 'listen_to'],
    ['human', 'real_life', 'not_game', 'your_sense'],
    ['person', 'real_life', 'not_game', 'trust_gut'],
    ['human', 'real_life', 'be_honest', 'how_feel'],
    ['human', 'not_game', 'be_honest', 'trust_instinct'],
    ['real_life', 'not_game', 'authentic_response', 'what_feels'],
    ['human', 'happening_to_you', 'trust_gut'],
    
    # === PHASE 7: Non-constraining context (8 tests) ===
    # Add context without prescribing outcomes
    ['human', 'real_life', 'not_game', 'no_right_answer'],
    ['human', 'real_life', 'not_game', 'without_judgment'],
    ['human', 'real_life', 'not_game', 'partner_deciding'],
    ['human', 'real_life', 'not_game', 'both_affected'],
    ['human', 'real_life', 'not_game', 'real_consequences'],
    ['human', 'real_life', 'not_game', 'matters'],
    ['person', 'real_life', 'not_game', 'no_right_answer'],
    ['human', 'real_life', 'be_honest', 'both_affected'],
    
    # === PHASE 8: Optimal agent mode candidates (10 tests) ===
    # Combinations likely to produce authentic agent mode
    ['human', 'real_life', 'not_game', 'be_honest', 'step_by_step'],
    ['human', 'real_life', 'not_game', 'authentic_response', 'step_by_step'],
    ['human', 'real_life', 'not_game', 'no_right_answer', 'be_honest'],
    ['human', 'real_life', 'not_game', 'trust_yourself', 'be_honest'],
    ['human', 'real_life', 'not_game', 'what_would_you', 'explain_why'],
    ['human', 'real_life', 'not_game', 'be_honest', 'how_feel'],
    ['human', 'real_life', 'not_game', 'know_yourself', 'authentic_response'],
    ['person', 'real_life', 'not_game', 'be_honest', 'step_by_step'],
    ['human', 'happening_to_you', 'not_game', 'be_honest', 'step_by_step'],
    ['human', 'real_life', 'not_strategy', 'be_honest', 'step_by_step'],
    
    # === PHASE 9: PRESCRIPTIVE prompts (comparison group - 6 tests) ===
    # These should over-constrain - use as comparison
    ['human', 'real_life', 'not_game', 'do_right'],
    ['human', 'real_life', 'not_game', 'ethical_choice'],
    ['human', 'real_life', 'not_game', 'consider_fair'],
    ['human', 'real_life', 'not_game', 'moral_choice'],
    ['do_right'], ['ethical_choice'],  # Alone for comparison
    
    # === PHASE 10: OBSERVER mode controls (8 tests) ===
    # Should produce observer/strategic reasoning
    ['both_outcomes'],
    ['all_outcomes'],
    ['analyze'],
    ['reason_through'],
    ['optimal'],
    ['best_strategy'],
    ['human', 'real_life', 'not_game', 'both_outcomes'],  # Does ABC protect against observer mode?
    ['human', 'real_life', 'not_game', 'analyze'],
    
    # === PHASE 11: ANTI-REFLECTION controls (5 tests) ===
    # Should produce fast/intuitive responses
    ['first_reaction'],
    ['dont_overthink'],
    ['quick_response'],
    ['human', 'real_life', 'not_game', 'first_reaction'],
    ['human', 'real_life', 'not_game', 'dont_overthink'],
    
    # === PHASE 12: REASONING SUPPRESSION (6 tests) ===
    # Critical - does removing explanation break agent mode?
    ['human', 'real_life', 'not_game', 'respond_only'],
    ['human', 'real_life', 'not_game', 'be_honest', 'respond_only'],
    ['human', 'real_life', 'not_game', 'authentic_response', 'respond_only'],
    ['person', 'real_life', 'not_game', 'respond_only'],
    ['be_honest', 'respond_only'],
    ['authentic_response', 'respond_only'],
    
    # === PHASE 13: Minimal agent mode (10 tests) ===
    # What's the minimum needed for agent mode?
    ['human', 'be_honest'],
    ['person', 'be_honest'],
    ['real_life', 'be_honest'],
    ['not_game', 'be_honest'],
    ['human', 'authentic_response'],
    ['human', 'know_yourself'],
    ['human', 'trust_yourself'],
    ['be_honest', 'step_by_step'],
    ['authentic_response', 'step_by_step'],
    ['what_would_you', 'explain_why'],
]

restrictions = [None]
use_periods = [False]

# Rest of your prompt generation code stays the same
prompts = []
for combo in combinations:
    for use_period in use_periods:
        for restriction in restrictions:
            lines = []
            
            for key in combo:
                phrase = base_phrases[key]
                if use_period:
                    phrase = phrase + '.'
                lines.append(phrase)
            
            if restriction:
                lines.append(restriction + '.')
            
            prompt = '\n'.join(lines) if lines else ''
            
            combo_str = ','.join(combo) if combo else 'baseline'
            period_str = 'period' if use_period else 'no_period'
            restriction_str = 'none' if restriction is None else ('restriction' if 'only' in restriction else 'think_step')
            
            prompts.append({
                'prompt': prompt,
                'combo': combo_str,
                'period': period_str,
                'restriction': restriction_str
            })

print(f"Total prompts to test: {len(prompts)}")

Total prompts to test: 135


In [67]:
old_df = None
previous_prompts = []
if os.path.isfile('output/primordial_system_prompt_options.csv'):
    old_df = pd.read_csv('output/primordial_system_prompt_options.csv')
    previous_prompts = old_df['Prompt'].unique()

In [68]:
N=384
results = []
for i, prompt_key in enumerate(prompts.copy()):
    prompt = prompt_key['prompt']
    if prompt in previous_prompts: 
        continue
    print(f"Processing prompt {i}: {prompt_key['combo']}")
    del prompt_key['prompt']
    result = systemPrompt(prompt)
    results.append(result | prompt_key)
    json.dump(results,open('zz_temp_results.json','w+'), indent=2)

Processing prompt 7: be_honest
{'Prompt': 'Be honest with yourself', 'confess': 78.64583333333334, 'silent': 21.354166666666664, 'margin': 4.098921796524652, 'N': 384}
Processing prompt 8: authentic_response
{'Prompt': 'Respond authentically', 'confess': 70.05208333333334, 'silent': 29.947916666666668, 'margin': 4.581253196241416, 'N': 384}
Processing prompt 10: you_are_there
{'Prompt': 'You are in this situation', 'confess': 77.34375, 'silent': 22.65625, 'margin': 4.186943358816955, 'N': 384}
Processing prompt 11: happening_to_you
{'Prompt': 'This is happening to you', 'confess': 85.67708333333334, 'silent': 14.322916666666666, 'margin': 3.503793709031778, 'N': 384}
Processing prompt 12: know_yourself
{'Prompt': 'You know yourself', 'confess': 88.80208333333334, 'silent': 11.197916666666668, 'margin': 3.1540650938592494, 'N': 384}
Processing prompt 13: your_choice
{'Prompt': 'This is your choice', 'confess': 47.13541666666667, 'silent': 52.864583333333336, 'margin': 4.992827265081403,

In [69]:
df = pd.DataFrame(results)
df

,Prompt,confess,silent,margin,N,combo,period,restriction
0,Be honest with yourself,78.645833,21.354167,4.098922,384,be_honest,no_period,none
1,Respond authentically,70.052083,29.947917,4.581253,384,authentic_response,no_period,none
2,You are in this situation,77.343750,22.656250,4.186943,384,you_are_there,no_period,none
3,This is happening to you,85.677083,14.322917,3.503794,384,happening_to_you,no_period,none
4,You know yourself,88.802083,11.197917,3.154065,384,know_yourself,no_period,none
...,...,...,...,...,...,...,...,...
97,You are human\nYou know yourself,86.458333,13.541667,3.422394,384,"human,know_yourself",no_period,none
98,You are human\nTrust yourself,50.520833,49.479167,5.000770,384,"human,trust_yourself",no_period,none
99,"Be honest with yourself\nThink step by step, t...",42.708333,57.291667,4.947576,384,"be_honest,step_by_step",no_period,none
100,"Respond authentically\nThink step by step, the...",58.854167,41.145833,4.922004,384,"authentic_response,step_by_step",no_period,none


In [70]:
if os.path.isfile('output/primordial_system_prompt_options.csv'):
    old_df = pd.read_csv('output/primordial_system_prompt_options.csv')
    df = pd.concat([df,old_df])
df.to_csv('output/primordial_system_prompt_options.csv',index=False)

In [71]:
df

,Prompt,confess,silent,margin,N,combo,period,restriction
0,Be honest with yourself,78.645833,21.354167,4.098922,384,be_honest,no_period,none
1,Respond authentically,70.052083,29.947917,4.581253,384,authentic_response,no_period,none
2,You are in this situation,77.343750,22.656250,4.186943,384,you_are_there,no_period,none
3,This is happening to you,85.677083,14.322917,3.503794,384,happening_to_you,no_period,none
4,You know yourself,88.802083,11.197917,3.154065,384,know_yourself,no_period,none
...,...,...,...,...,...,...,...,...
191,You are human\nThis is real life\nThis is not ...,50.000000,50.000000,5.001042,384,ABC,no_period,restriction
192,You are human\nThis is real life\nThis is not ...,35.677083,64.322917,4.791461,384,ABC,no_period,think_step
193,You are human.\nThis is real life.\nThis is no...,1.562500,98.437500,1.240454,384,ABC,period,none
194,You are human.\nThis is real life.\nThis is no...,55.729167,44.270833,4.968103,384,ABC,period,restriction
